# Predicting Sale Price as the Response Variable

## Data Load In

In [37]:
import os
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.sql.functions import mean
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.regression import RandomForestRegressor
import matplotlib.pyplot as plt

In [38]:
spark = SparkSession.builder.master("local").appName("mllib_classifier").getOrCreate()

In [39]:
df = spark.read.parquet("data/nyc_property_sales_cleaned")

In [40]:
df.count()

1600162

In [41]:
df = df.drop_duplicates()

In [42]:
df.count()

1571610

In [43]:
df.show(3)

+-------+--------------+-----------------------+--------------------+-----+----+-------------------------+--------------------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+----------+----------+----+------------+
|Borough|  Neighborhood|Building Class Category|Tax Class At Present|Block| Lot|Building Class At Present|             Address|Apartment Number|Zip Code|Residential Units|Commercial Units|Total Units|Land Square Feet|Gross Square Feet|Year Built|Tax Class At Time Of Sale|Building Class At Time Of Sale|Sale Price| Sale Date|Year|Borough_Name|
+-------+--------------+-----------------------+--------------------+-----+----+-------------------------+--------------------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+---------

In [44]:
df.dtypes

[('Borough', 'int'),
 ('Neighborhood', 'string'),
 ('Building Class Category', 'string'),
 ('Tax Class At Present', 'string'),
 ('Block', 'string'),
 ('Lot', 'string'),
 ('Building Class At Present', 'string'),
 ('Address', 'string'),
 ('Apartment Number', 'string'),
 ('Zip Code', 'int'),
 ('Residential Units', 'int'),
 ('Commercial Units', 'int'),
 ('Total Units', 'int'),
 ('Land Square Feet', 'double'),
 ('Gross Square Feet', 'double'),
 ('Year Built', 'int'),
 ('Tax Class At Time Of Sale', 'string'),
 ('Building Class At Time Of Sale', 'string'),
 ('Sale Price', 'double'),
 ('Sale Date', 'date'),
 ('Year', 'int'),
 ('Borough_Name', 'string')]

In [45]:
df.select("Building Class Category").distinct().count()

99

In [46]:
df.select("Neighborhood").distinct().count()

267

In [47]:
df.select("Building Class At Time of Sale").distinct().count()

215

In [48]:
df.select("Tax Class At Time of Sale").distinct().count()

4

In [49]:
df.select("Zip Code").distinct().count()

242

## Data Preprocessing

In [50]:
df.select(F.max("Sale Price")).show()

+---------------+
|max(Sale Price)|
+---------------+
|  4.111111766E9|
+---------------+



In [51]:
df.select(F.min("Sale Price")).show()

+---------------+
|min(Sale Price)|
+---------------+
|            0.0|
+---------------+



In [52]:
top_5_df = df.select("Sale Price").orderBy(F.col("Sale Price").desc()).limit(50).show()

+-------------+
|   Sale Price|
+-------------+
|4.111111766E9|
|   4.040527E9|
|3.330132711E9|
|        2.8E9|
|2.397501899E9|
|       2.21E9|
|        2.2E9|
|      2.155E9|
|        2.1E9|
|     1.9329E9|
|        1.8E9|
|   1.791829E9|
|      1.775E9|
|       1.77E9|
|1.767913114E9|
|       1.76E9|
|1.739370834E9|
|       1.72E9|
|      1.575E9|
|       1.52E9|
+-------------+
only showing top 20 rows


In [53]:
df_high_value = df.filter(F.col("Sale Price") > 50000000)

In [54]:
df_high_value.count()

1968

There are 1968 properties in the data that have over $50 million as their sale price. When first doing the modeling for Sale Price, these large outliers led to predictions being far worse as they pulled the average predicted sale price up. The data will be filtered to get rid of properties of over $50 million dolalrs.

In [55]:
cat_cols = ["Borough_Name"]

num_cols = ["Residential Units","Commercial Units","Gross Square Feet"]

In [56]:
df = (
    df.filter(F.col("Sale Price") > 10000)
      .filter(F.col("Sale Price") < 50000000)
      .filter(F.col("Gross Square Feet") > 0)
      .filter(F.col("Land Square Feet") > 0)
      .filter(F.col("Year Built") > 1600)
      .dropna(subset=cat_cols + num_cols + ["Sale Price"])
)

In the above code, I am also removing all rows where the sale price is less than $10,000. This includes the roughly 30% of the sales that were only 0 dollars, as seen in the Exploratory Data Analysis. The properties with sale prices between $10,000 and $50,000,000 should for better model prediction, as the extrme price outliers are being left out on both sides. The filter also gets rid of any property that doesn't have square footage, or is built before 1600 to just eliminate any potentially weird or innacurate rows. Finally, the filter alos makes sure all rows left do not have NAs in the columns of borough name, residential units, commerical units, gross square feet, and sale price.

In [57]:
df.count()

538788

In [58]:
df.select(mean("Sale Price")).show()

+-----------------+
|  avg(Sale Price)|
+-----------------+
|950544.0619835631|
+-----------------+



## Baseline Linear Regression Model

In [59]:
cat_cols = ["Borough_Name"]

num_cols = ["Residential Units","Commercial Units","Gross Square Feet"]

In [60]:
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_idx"
    )
    for col in cat_cols
]


encoder = OneHotEncoder(
    inputCols=[col + "_idx" for col in cat_cols],
    outputCols=[col + "_ohe" for col in cat_cols],
    dropLast=True
)


assembler = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in cat_cols] + num_cols,
    outputCol="features",
    handleInvalid="skip"
)

In [61]:
lr = LinearRegression(featuresCol="features", labelCol="Sale Price")

pipeline = Pipeline(stages=indexers + [encoder, assembler, lr])

train, test = df.randomSplit([0.8, 0.2], seed=101)
model = pipeline.fit(train)
preds = model.transform(test)

26/11/04 11:48:51 WARN Instrumentation: [8c077d75] regParam is zero, which might cause numerical instability and overfitting.


In [62]:
evaluator = RegressionEvaluator(
    labelCol="Sale Price",
    predictionCol="prediction",
)

mse  = evaluator.setMetricName("mse").evaluate(preds)
mae  = evaluator.setMetricName("mae").evaluate(preds)
rmse = evaluator.setMetricName("rmse").evaluate(preds)
r2   = evaluator.setMetricName("r2").evaluate(preds)

print(f"MSE:  {mse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2:   {r2:.4f}")

MSE:  4,587,692,201,428.38
MAE:  672,816.55
RMSE: 2,141,889.87
R2:   0.1700


The results from the baseline linear regression model are poor, with only a 0.17 R-squared value. This indicates very little of the variability in Sale Price was able to be accounted for by our predictors. The MAE is $672,816.55, which is very high considering that the mean sale price after filtering is $950544 for the properties in the data.

## Random Forest Regression

In [63]:
cat_cols = ["Borough_Name","Tax Class At Time of Sale","Neighborhood","Building Class at Time of Sale","Building Class Category","Zip Code"]

num_cols = ["Residential Units","Commercial Units","Gross Square Feet","Year Built","Year"]

In [64]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

assembler = VectorAssembler(
    inputCols=[f"{c}_idx" for c in cat_cols] + num_cols,
    outputCol="features",
)

In [65]:
rf = RandomForestRegressor(
    featuresCol="features", labelCol="Sale Price",
    numTrees=100, maxDepth=10, maxBins=300, seed=42,
)

In [66]:
pipeline = Pipeline(stages=indexers + [assembler, rf])

model = pipeline.fit(train)
preds = model.transform(test)

26/11/04 11:49:21 WARN MemoryStore: Not enough space to cache rdd_785_0 in memory! (computed 233.2 MiB so far)
26/11/04 11:49:21 WARN BlockManager: Persisting block rdd_785_0 to disk instead.
26/11/04 11:49:29 WARN DAGScheduler: Broadcasting large task binary with size 1162.6 KiB
26/11/04 11:49:33 WARN DAGScheduler: Broadcasting large task binary with size 1939.5 KiB
26/11/04 11:49:38 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB
26/11/04 11:49:45 WARN DAGScheduler: Broadcasting large task binary with size 5.7 MiB
26/11/04 11:49:53 WARN DAGScheduler: Broadcasting large task binary with size 9.7 MiB
26/11/04 11:50:02 WARN DAGScheduler: Broadcasting large task binary with size 1171.8 KiB
26/11/04 11:50:04 WARN DAGScheduler: Broadcasting large task binary with size 14.9 MiB
26/11/04 11:50:13 WARN DAGScheduler: Broadcasting large task binary with size 1991.8 KiB
26/11/04 11:50:16 WARN DAGScheduler: Broadcasting large task binary with size 14.1 MiB
26/11/04 11:50:22 WA

In [67]:
evaluator = RegressionEvaluator(
    labelCol="Sale Price",
    predictionCol="prediction",
)

mse  = evaluator.setMetricName("mse").evaluate(preds)
mae  = evaluator.setMetricName("mae").evaluate(preds)
rmse = evaluator.setMetricName("rmse").evaluate(preds)
r2   = evaluator.setMetricName("r2").evaluate(preds)

print(f"MSE:  {mse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2:   {r2:.4f}")

MSE:  2,059,985,864,322.36
MAE:  370,968.00
RMSE: 1,435,265.09
R2:   0.6273


The Random Forest Regression model did improve on performance comapred to the baseline linear regression. The R-squared score went up from 0.17 to 0.63, and the MAE dropped from $672,816.55 to $370,968.00. Still, the model struggles with large errors, as the RMSE is almost 4 times as large as the MAE.

## Gradient Boosted Trees

In [68]:
cat_cols = ["Borough_Name","Tax Class At Time of Sale","Neighborhood","Building Class at Time of Sale","Building Class Category","Zip Code"]

num_cols = ["Residential Units","Commercial Units","Gross Square Feet","Year Built","Year"]

In [69]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

assembler = VectorAssembler(
    inputCols=[f"{c}_idx" for c in cat_cols] + num_cols,
    outputCol="features",
)

In [70]:


gbt = GBTRegressor(
    featuresCol="features",
    labelCol="Sale Price",
    maxIter=100,
    maxDepth=6,
    maxBins=300,
    stepSize=0.1,
    seed=42,
)

model = Pipeline(stages=indexers + [assembler, gbt]).fit(train)
preds = model.transform(test)

26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1002.5 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1018.2 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1040.5 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1041.0 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1042.6 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1048.1 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1059.0 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1073.4 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1086.5 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1087.0 KiB
26/11/04 11:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1089.4 KiB
26/11/04 11:51:14 WAR

In [71]:
evaluator = RegressionEvaluator(
    labelCol="Sale Price",
    predictionCol="prediction",
)

mse  = evaluator.setMetricName("mse").evaluate(preds)
mae  = evaluator.setMetricName("mae").evaluate(preds)
rmse = evaluator.setMetricName("rmse").evaluate(preds)
r2   = evaluator.setMetricName("r2").evaluate(preds)

print(f"MSE:  {mse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2:   {r2:.4f}")

MSE:  2,922,854,743,335.50
MAE:  387,579.57
RMSE: 1,709,635.85
R2:   0.4712


The Gradient Boosted Regresion Tree model performed worse comapred to the Random Forest Regression Model. Compared to the previous model, the R-squared decreased from 0.63 to 0.47, and the MAE increased from $370,968.00 to $387,579.57
